# Climate Data Cleaning: PRISM (point-sampled at WQ stations)

Cleans the PRISM daily climate extract — gridded climate sampled **at each EPA
water-quality monitoring location** — into a tidy table keyed on `station_id` +
`date`, ready to join straight onto the EPA samples.

**Input:**  `data/tabular/01_raw/climate/prism-iowa-climate.csv`
**Output:** `data/tabular/02_clean/climate/prism-iowa-climate-clean.csv`

**Why this source is different from ISU** — the companion ISU climate feed
(`climate-clean.ipynb`) is keyed on ISU station codes (e.g. `IA1533`) and has to
be spatially matched to each water-quality site via nearest-neighbour search.
PRISM here was already sampled **at the WQ station coordinates**: every
`station_id` is an EPA WQX `MonitoringLocationIdentifier` (e.g. `USGS-05387490`),
and **100% of the cleaned WQ stations are covered**. So the merge is a *direct*
key join on `station_id` + `date` — no spatial matching, no distance error.

**Raw variables** (PRISM native units): `tmax`, `tmin`, `tdmean` are daily
max / min air temperature and mean dewpoint in **°C**; `ppt` is daily
precipitation in **mm**.

**Pipeline**
1. Load the raw extract.
2. **Fix types, sentinels & join key** — parse `date`, coerce numerics, map
   PRISM nodata (`-9999`) to `NaN`, drop rows with no `station_id`/`date`, and
   enforce one row per `(station_id, date)` so the join can't fan out.
3. **Range-validate** each variable against generous physical bounds.
4. **Cross-field consistency** — null `(tmin, tmax)` pairs where `tmin > tmax`.
5. **Rename to self-documenting, unit-suffixed `prism_*` columns** so the
   provenance and units are explicit and nothing collides with the ISU climate
   features in the merge.
6. Sanity-check and save.

The raw file is already exceptionally clean (a complete
`1,666 stations × 2,464 days` grid with no duplicates, no `-9999` sentinels, and
only a handful of whole-day grid gaps), so steps 2–4 mostly **guard against
future re-pulls** rather than fixing current damage. The lasting value is the
robust paths and the clear, units-bearing column contract.

> **Path note:** like the other migrated cleaners this reads `01_raw` and writes
> `02_clean`. There is no PRISM merge step yet; when one is added it should read
> from `02_clean/climate/` and join `station_id` -> `MonitoringLocationIdentifier`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "climate"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "climate"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

## Step 1 — Load

~4.1M rows. We read `station_id` as a string dtype up front (it's a categorical
identifier, never arithmetic) and leave the four measurements as floats.

In [ ]:
VALUE_COLS = ["tmax", "tmin", "ppt", "tdmean"]

df = pd.read_csv(RAW_DIR / "prism-iowa-climate.csv", dtype={"station_id": "string"})
n_raw = len(df)
print(f"Loaded {n_raw:,} station-day rows across {df['station_id'].nunique():,} stations")
df.head()

## Step 2 — Fix types, sentinels & enforce the join key

Parse `date`, coerce the measurements to numeric, and map PRISM's `-9999` nodata
sentinel to `NaN` (none are present today, but a future re-pull could include
them). Drop rows with no `station_id`/`date` (they can't join), then **guarantee
one row per `(station_id, date)`** so the downstream join can't silently
multiply water-quality rows.

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
for col in VALUE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# PRISM nodata sentinel -> NaN (defensive; none in the current extract).
n_sentinel = int((df[VALUE_COLS] == -9999).sum().sum())
df[VALUE_COLS] = df[VALUE_COLS].replace(-9999, np.nan)
print(f"PRISM -9999 sentinels mapped to NaN: {n_sentinel:,}")

before = len(df)
df = df[df["station_id"].notna() & df["date"].notna()].copy()
print(f"Dropped rows with missing station_id/date: {before - len(df):,}  ({before:,} -> {len(df):,})")

n_dupes = df.duplicated(["station_id", "date"]).sum()
if n_dupes:
    print(f"Collapsing {n_dupes:,} duplicate (station_id, date) rows via mean")
    df = df.groupby(["station_id", "date"], as_index=False)[VALUE_COLS].mean()
else:
    print("(station_id, date) is already unique — no collapsing needed")
assert not df.duplicated(["station_id", "date"]).any()

## Step 3 — Range validation

Null any value outside generous physical bounds (°C for temperatures/dewpoint,
mm/day for precipitation). Bounds are wide Iowa extremes — the aim is to catch
sentinel/decoding garbage, not to trim real tails.

In [ ]:
# column : (low, high) physically plausible bounds
RANGES = {
    "tmax": (-60, 60),
    "tmin": (-60, 60),
    "tdmean": (-60, 60),
    "ppt": (0, 500),
}
total_nulled = 0
for col, (lo, hi) in RANGES.items():
    bad = df[col].notna() & ~df[col].between(lo, hi)
    if bad.sum():
        print(f"{col:<8} nulled {int(bad.sum()):>4} value(s) outside [{lo}, {hi}]")
    df.loc[bad, col] = np.nan
    total_nulled += int(bad.sum())
print(f"Total out-of-range values nulled: {total_nulled:,}")

## Step 4 — Cross-field consistency

A day's minimum temperature cannot exceed its maximum. Where it does, both ends
are unreliable, so we null the pair.

In [ ]:
bad = df["tmin"].notna() & df["tmax"].notna() & (df["tmin"] > df["tmax"])
print(f"tmin > tmax: nulled {int(bad.sum())} inconsistent pair(s)")
df.loc[bad, ["tmin", "tmax"]] = np.nan

## Step 5 — Rename to a self-documenting column contract

The raw names (`tmax`, `tmin`, `ppt`, `tdmean`) are opaque and unitless, and the
temperature/precip names would collide with the ISU climate features once both
are merged onto the water-quality table. We rename to unit-suffixed `prism_*`
columns so units and provenance are explicit and unambiguous. The `station_id` +
`date` key keeps its native names (it joins directly to
`MonitoringLocationIdentifier`).

In [ ]:
RENAME = {
    "tmax": "prism_tmax_c",
    "tmin": "prism_tmin_c",
    "ppt": "prism_ppt_mm",
    "tdmean": "prism_tdmean_c",
}
df = df.rename(columns=RENAME)
OUTPUT_COLS = ["station_id", "date"] + list(RENAME.values())
df = df[OUTPUT_COLS]
df.head()

## Step 6 — Sanity check

Confirm the join key is unique, the grid is intact, and the values sit in
sensible ranges. Missing values are left as `NaN` (the only gaps are a few whole
PRISM grid-days); they're handled at modeling time rather than fabricated here.

In [ ]:
print(f"Rows: {len(df):,} ({len(df) / n_raw:.0%} of raw)  |  "
      f"stations: {df['station_id'].nunique():,}  |  days: {df['date'].nunique():,}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}\n")
miss = df[list(RENAME.values())].isna().sum()
print("Missing per column:")
print(pd.concat([miss.rename("missing"),
                 (miss / len(df) * 100).round(2).rename("pct")], axis=1).to_string())
print()
df[list(RENAME.values())].describe().round(2)

## Step 7 — Save

`date` is written as `YYYY-MM-DD` so it joins cleanly against the EPA sample
date string in the merge step.

In [ ]:
df["date"] = df["date"].dt.strftime("%Y-%m-%d")
out_file = CLEAN_DIR / "prism-iowa-climate-clean.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} station-day rows -> {out_file}")